# 1. Imports + basic setup

In [ ]:
import os
import json
import math
import random
from dataclasses import dataclass, asdict
from typing import Dict, Optional, Any

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())


torch: 2.10.0+cu128
cuda available: True


# 2. Reproducibility helpers + device

In [ ]:
def set_seed(seed: int = 42):
    """Set random seeds for Python, NumPy, and PyTorch to ensure reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

# 3. Load frozen encoder

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

encoder_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(encoder_name)
encoder = AutoModel.from_pretrained(encoder_name).to(device)
encoder.eval()

for p in encoder.parameters():
    p.requires_grad = False

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# 4. Function to extract CLS embeddings

In [ ]:
@torch.no_grad()
def get_cls_embeddings(texts, tokenizer, encoder, device, max_length=128, batch_size=64):
    """Extract [CLS] token embeddings from a transformer encoder for a list of texts.

    Processes inputs in batches with no gradient computation. Returns the
    first-token hidden state (768-d for DistilBERT) as a NumPy array.

    Args:
        texts: List of input strings.
        tokenizer: HuggingFace tokenizer.
        encoder: HuggingFace transformer model (e.g. DistilBERT).
        device: torch.device for inference.
        max_length: Maximum token length per input.
        batch_size: Number of texts per forward pass.

    Returns:
        np.ndarray of shape (len(texts), hidden_dim) with CLS embeddings.
    """
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        enc = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )

        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        out = encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   # [batch, 768]

        embeddings.append(cls.cpu().numpy())

    return np.vstack(embeddings)

# 5. Load Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load CSVs + minimal cleaning

In [ ]:
import shutil

BASE_PATH = "/content/drive/MyDrive/Lokaverkefni - Gögn"

# filenames are changed if the normalized data is being used for training and testing
train_df = pd.read_csv(os.path.join(BASE_PATH, "train_sample.csv"))
val_df   = pd.read_csv(os.path.join(BASE_PATH, "validation_sample.csv"))
test_df  = pd.read_csv(os.path.join(BASE_PATH, "test_sample.csv"))

print("raw shapes:", train_df.shape, val_df.shape, test_df.shape)
train_df.head()

raw shapes: (85675, 8) (18359, 8) (18360, 8)


,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description
0,1288696,Rae Dunn Mixing Bowls Reserved K. Cannon,1,Home/Kitchen & Dining/Dining & Entertaining,Rae Dunn,100.0,0,Reserved for Kim Cannon
1,1016811,2 4x4 monogrammed decal/sticker,1,Handmade/Paper Goods/Sticker,NaN,8.0,1,Monogram decal available and any size! Please ...
2,5636,CINDY CRAWFORD MEANINGFUL BEAUTY,1,Beauty/Skin Care/Face,NaN,30.0,0,Cindy Crawford meaningful beauty 5pcs Toner ha...
3,1469134,Simply Southern,2,Women/Tops & Blouses/T-Shirts,Simply Southern,18.0,1,T-Shirts Size Small. Barley worn. Card holder ...
4,455146,⚡️NWT Just do it Leggings Size Medium,1,"Women/Athletic Apparel/Pants, Tights, Leggings",Nike,34.0,0,New with tag Nike Camo Leg-A-See. Original pri...


# 6. Extract each field separately

In [ ]:
EMB_MAX_LEN = 64

def make_text(df):
    """Extract the item_description column from a DataFrame as a list of strings.

    Handles missing or NaN values by replacing them with empty strings,
    and strips leading/trailing whitespace from each entry.

    Args:
        df: DataFrame expected to contain an 'item_description' column.

    Returns:
        List of cleaned description strings, one per row.
    """
    return df.apply(
      lambda r: str(r.get('item_description', '') or '').strip(),
      axis=1
    ).tolist()

desc_train = get_cls_embeddings(make_text(train_df), tokenizer, encoder, device, max_length=EMB_MAX_LEN)
desc_val   = get_cls_embeddings(make_text(val_df),   tokenizer, encoder, device, max_length=EMB_MAX_LEN)
desc_test  = get_cls_embeddings(make_text(test_df),  tokenizer, encoder, device, max_length=EMB_MAX_LEN)

# 7. Concatenate into XGBoost input

In [ ]:
X_train = desc_train
X_val   = desc_val
X_test  = desc_test

y_train = np.log1p(train_df["price"].values)
y_val   = np.log1p(val_df["price"].values)
y_test  = np.log1p(test_df["price"].values)

# 8. Train XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42
)

xgb.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

# 9. Evaluate

In [ ]:
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Predict on held-out test split
pred_log_test = xgb.predict(X_test)
pred_test     = np.expm1(pred_log_test)

# Test RMSLE (the headline metric)
rmsle_test = np.sqrt(np.mean((pred_log_test - y_test) ** 2))
mae_test   = mean_absolute_error(test_df["price"].values, pred_test)
rmse_test  = np.sqrt(mean_squared_error(test_df["price"].values, pred_test))

print(f"TEST  RMSLE: {rmsle_test:.4f}")
print(f"TEST  MAE:   {mae_test:.4f}")
print(f"TEST  RMSE:  {rmse_test:.4f}")

# Save predictions for cross-condition comparison and error analysis
CONDITION = "p3"
MODEL_TAG = "xgb_cls_desc"

OUT_DIR = "/content/drive/MyDrive/Lokaverkefni - Gögn/Predictions"
os.makedirs(OUT_DIR, exist_ok=True)

test_predictions = pd.DataFrame({
    "train_id":     test_df["train_id"].values,
    "y_true_log":   y_test,
    "y_pred_log":   pred_log_test,
    "y_true_price": test_df["price"].values,
    "y_pred_price": pred_test,
})

out_path = os.path.join(OUT_DIR, f"test_predictions_{MODEL_TAG}_{CONDITION}.csv")
test_predictions.to_csv(out_path, index=False)
print(f"Saved {len(test_predictions)} predictions → {out_path}")

# Save metrics summary alongside
metrics_path = os.path.join(OUT_DIR, f"metrics_{MODEL_TAG}_{CONDITION}.json")
with open(metrics_path, "w") as f:
    json.dump({
        "model": MODEL_TAG,
        "condition": CONDITION,
        "split": "test",
        "n": int(len(test_predictions)),
        "rmsle": float(rmsle_test),
        "mae":   float(mae_test),
        "rmse":  float(rmse_test),
        "emb_max_length": EMB_MAX_LEN,
    }, f, indent=2)
print(f"Saved metrics → {metrics_path}")

TEST  RMSLE: 0.6159
TEST  MAE:   13.9821
TEST  RMSE:  35.0762
Saved 18360 predictions → /content/drive/MyDrive/Lokaverkefni - Gögn/Predictions/test_predictions_xgb_cls_desc_p3.csv
Saved metrics → /content/drive/MyDrive/Lokaverkefni - Gögn/Predictions/metrics_xgb_cls_desc_p3.json
